In [ ]:
import simpy
import pandas as pd         
import numpy as np          
import random               
from itertools import combinations  # 주문 내에서 함께 출고되는 부품쌍 생성 (Kitting Slotting에 사용)
from collections import Counter     # 함께 출고되는 부품쌍의 출현 빈도 계산 (어떤 부품을 서로 가깝게 배치할지 판단하기 위함)

# --- 시스템 환경 설정 (상수 정의) ---
# 창고의 물리적 레이아웃과 물류 로봇 성능 정의
RACK_START_Y, RACK_END_Y = 4.0, 15.1       # 랙이 설치된 Y축의 시작과 끝 좌표 범위 (4.0m ~ 15.1m 사이에만 적재 가능)
AISLE_X = [1.2, 3.6, 6.0, 8.4, 10.8, 13.2] # 창고 통로(Aisle)가 위치한 X축 좌표 리스트 (수정 혹은 검토 필요: "부품은 이 통로 중 하나에 배치됨")
STATIONS = {'S01': [3.5, 0.5, 0], 'S02': [11.0, 0.5, 0]} # 작업 스테이션 위치 (x, y, z), CTU가 작업 후 반드시 돌아오는 목적지)
CTU_SPEED_XY = 1.8  # 로봇의 XY 이동 속도 (m/s)
CTU_SPEED_Z = 0.5   # 로봇의 리프트(Z) 상하 이동 속도 (m/s)
TOTE_CAPACITY = 7   # CTU가 한 번에 운반할 수 있는 최대 토트 박스 수 (수정 혹은 검토 필요: "토트 박스 용량" -> 7층 CTU가 운반할 수 있는 최대 토트 박스 수)
SHELF_HEIGHTS = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0] # 선반의 층별 높이(Z축) 리스트 (부품의 Z좌표로 사용)

# ==========================================
# CTU class 물류 로봇
# ==========================================
class CTU: # 물류 로봇 정의
    def __init__(self, env, id, warehouse, dispatch_rule): # env: 시뮬레이션 환경, id: CTU 번호, warehouse: 작업정보 접근, dispatch_rule: 작업 할당 알고리즘)
        self.env = env                  # Simpy 시뮬레이션 환경 객체 (현재 시뮬레이션 시간을 알고 있음)
        self.id = id                    # 로봇의 고유 번호 (0, 1, 2...)
        self.warehouse = warehouse      # 로봇이 소속된 창고 객체
        self.dispatch_rule = dispatch_rule # 이 로봇에게 적용될 작업 할당 알고리즘 (nearest, density, load_balance)
        
        # 초기 로봇 위치 설정: 로봇 2대가 서로 다른 스테이션에서 시작하도록 설정.
        self.pos = STATIONS['S01'] if id == 0 else STATIONS['S02'] # 0번 CTU는 S01에서 시작, 1번 CTU는 S02에서 시작

        # 총 이동거리 기록
        # 처리 작업 수 기록
        self.total_distance = 0         # 시뮬레이션 동안 로봇이 이동한 총 누적 거리
        self.completed_jobs = 0         # 처리한 총 작업 수 (현재 코드에서는 집계만 하고 사용 안 함)
        self.busy = False               # 로봇이 현재 작업 중인지 여부 (True: 작업중, False: 대기중)
        
        # 핵심: env.process(self.run())
        # CTU의 작업 수행 프로세스를 시뮬레이션에 등록
        # run() 함수가 시간 흐름에 따라 실행
        # 이 객체가 생성되는 순간, self.run() 메서드를 SimPy의 프로세스로 등록
        # 시뮬레이션이 시작(env.run)되면 로봇은 독립적으로 움직임.
        self.process = env.process(self.run())

    # 이동 시간 및 거리 계산
    # 목표 위치까지 이동시간 계산 함수
    def calculate_travel_time(self, target_pos):
        # 현재 위치(self.pos)와 목표 위치(target_pos) 사이의 좌표 차이 -> 거리 절댓값 계산
        dx = abs(target_pos[0] - self.pos[0]) # X축 이동거리
        dy = abs(target_pos[1] - self.pos[1]) # Y축 이동거리
        dz = abs(target_pos[2] - self.pos[2]) # Z축 이동거리
        
        # 이동 시간 계산 로직:
        # 로봇은 XY 평면 이동과 Z축(리프트) 상승/하강을 '동시에' 할 수 있다고 가정
        # XY 이동 시간과 Z 이동 시간 중 더 오래 걸리는 시간이 최종 소요 시간(max)이 됩니다.
        travel_time = max((dx + dy) / CTU_SPEED_XY, dz / CTU_SPEED_Z)
        
        # 이동시간 + 총 이동거리 반환
        # 총 이동 거리: 맨해튼 거리 방식(직각 이동)으로 계산하여 반환 -> 수정/검토 사항: dx + dy or dx + dy + dz
        return travel_time, (dx + dy + dz)


    # CTU의 실제 작업 수행 프로세스
    def run(self): 
        while True: # 시뮬레이션이 끝날 때까지 계속 반복
            # 1. 창고에 처리해야 할 일(job_queue)이 있는지 확인
            # 처리할 작업이 존재하면 '작업중' 상태 변경
            if self.warehouse.job_queue:
                self.busy = True # 작업 시작 상태로 변경
                
                # 2. 작업 할당 규칙에 따라 warehouse에 작업 묶음(batch)을 요청
                # 어떤 작업을 가져올지는 dispatch_rule에 따라 창고가 결정
                batch = self.warehouse.apply_dispatching(self, self.dispatch_rule)
                
                if batch:
                    # 3. 경로 최적화 (Greedy TSP)
                    # 할당받은 작업(batch)들 중, 현재 위치 기준 가까운 순서대로 방문 순서를 정렬
                    # np.linalg.norm: 벡터 간 유클리드 거리를 계산하는 함수
                    batch.sort(key=lambda j: np.linalg.norm(np.array(self.pos) - np.array(j['pos'])))
                    
                    # 4. 각 상품 위치로 이동하며 피킹 수행
                    for job in batch:
                        # 목표 위치까지 걸리는 시간(t)과 거리(d) 계산
                        t, d = self.calculate_travel_time(job['pos'])
                        
                        # "yield env.timeout(t)"
                        # 로봇이 이동하는 시간(t초) 동안 프로세스를 멈추고 대기
                        # 실제 이동시간만큼 시뮬레이션 시간 흐름 발생
                        yield self.env.timeout(t)
                        
                        self.pos = job['pos']     # 로봇 위치를 목표 지점으로 업데이트
                        self.total_distance += d  # 이동 거리 누적
                    
                    # 5. 모든 상품 피킹 완료 후, 가까운 스테이션으로 복귀
                    # d1: 현재 위치에서 S01까지의 거리
                    d1 = np.linalg.norm(np.array(self.pos[:2]) - np.array(STATIONS['S01'][:2]))
                    # d2: 현재 위치에서 S02까지의 거리
                    d2 = np.linalg.norm(np.array(self.pos[:2]) - np.array(STATIONS['S02'][:2]))
                    
                    # 더 가까운 스테이션 선택
                    near = STATIONS['S01'] if d1 < d2 else STATIONS['S02']
                    
                    # 선택된 스테이션까지 이동시간 및 거리 계산
                    t, d = self.calculate_travel_time(near)
                    yield self.env.timeout(t) # 이동 시간만큼 시뮬레이션 시간 흐름 
                    self.pos = near           # 위치 업데이트
                    self.total_distance += d  # 이동거리 누적
                
                self.busy = False # 작업 완료 후 상태를 대기중(idle)으로 변경
            
            else:
                # 6. 할 일이 없으면 1초 동안 대기했다가 다시 큐를 확인 (Busy Waiting 방지)
                yield self.env.timeout(1)


# ==========================================
# 창고 시스템 클래스 정의
# 재고 위치 및 작업 큐 관리 역할
# ==========================================
class Warehouse:
    def __init__(self, env, inventory, plan_df):
        self.env = env             # 시뮬레이션 환경 저장
        self.inventory = inventory # {부품코드: [x, y, z]} 형태의 딕셔너리 (부품별 저장 위치 정보, Slotting 결과)
        self.plan_df = plan_df     # 해당 날짜에 처리해야 할 출고 계획 데이터프레임
        self.job_queue = []        # 실시간으로 들어오는 주문들이 쌓이는 대기열, 아직 처리되지 않은 Picking 작업 큐

 # dispatching rule에 따라 CTU에게 할당할 작업 배치 생성 [cite: 54, 60]
    def apply_dispatching(self, ctu, rule):
        # 대기 중인 작업이 없으면 빈 리스트 반환
        if not self.job_queue: return []

        # ==========================================
        # 1. Nearest-seeded batch [cite: 64]
        # ==========================================
        if rule == 'nearest':
            # 현재 위치에서 가장 물리적 거리가 가까운 작업 순서로 정렬 (Nearest seed job 하나 선택) [cite: 65]
            self.job_queue.sort(key=lambda j: np.linalg.norm(np.array(ctu.pos) - np.array(j['pos'])))
            
            # 첫 번째 작업(Nearest seed)을 기준으로 삼음 [cite: 65]
            batch = [self.job_queue.pop(0)] 
            
            if self.job_queue:
                # Seed 주변에서 경로 상 가까운 job을 추가하기 위해 다시 정렬 [cite: 66]
                self.job_queue.sort(key=lambda j: np.linalg.norm(np.array(batch[0]['pos']) - np.array(j['pos'])))
                
                # 한 번에 최대 7개 box 로드 가능 [cite: 46]
                extra_count = min(len(self.job_queue), TOTE_CAPACITY - 1)
                
                # Q개가 될 때까지 추가하여 할당 및 큐에서 제거 [cite: 66]
                batch.extend(self.job_queue[:extra_count])
                del self.job_queue[:extra_count]
                
            return batch

        # ==========================================
        # 2. Density-seeded batch [cite: 67]
        # ==========================================
        elif rule == 'density':
            # 후보 seed 확인: 아직 안 끝난 job 중에서 MLTU 기준으로 가까운 순 Top-K개 확인 (K = 10 정도) [cite: 68, 69]
            K = min(10, len(self.job_queue)) 
            candidate_seeds = sorted(self.job_queue, key=lambda j: np.linalg.norm(np.array(ctu.pos) - np.array(j['pos'])))[:K]
            
            best_seed = None
            max_density = -1
            r_y = 2.65 # 1개 베이 길이 (m) 적용 (업라이트 및 박스 여유 공간 50mm 포함된 추정 길이) [cite: 3]
            
            # Seed 마다 밀집도 계산 [cite: 70]
            for seed in candidate_seeds:
                density = 0
                for job in self.job_queue:
                    # 1. 동일 복도(Aisle)에 있는지 확인하여 랙 관통 탐색을 철저히 방지 (폭 1200mm 제약 고려) [cite: 22, 24]
                    if job['pos'][0] == seed['pos'][0]:
                        # 2. 같은 복도라면, Seed 주변 반경 r안에 대기 job이 몇 개 있는지 세서 밀집도 정량화 [cite: 71]
                        dy = abs(job['pos'][1] - seed['pos'][1])
                        if dy <= r_y:
                            density += 1
                
                # 가장 밀집도가 큰 seed 갱신 [cite: 72]
                if density > max_density:
                    max_density = density
                    best_seed = seed
                    
            # 가장 밀집도가 큰 seed를 선택하여 nearest로 batch 만들어 할당 [cite: 72]
            # 선택된 best_seed를 기준으로 전체 대기열을 가까운 순으로 정렬
            self.job_queue.sort(key=lambda j: np.linalg.norm(np.array(best_seed['pos']) - np.array(j['pos'])))
            
            # CTU가 한 번에 로드 가능한 최대 7개 할당 [cite: 46]
            batch_size = min(len(self.job_queue), TOTE_CAPACITY)
            best_batch = self.job_queue[:batch_size]
            
            # 할당된 작업들은 큐에서 일괄 제거
            del self.job_queue[:batch_size]
            
            return best_batch

        # ==========================================
        # 3. Load-balance batch [cite: 73]
        # ==========================================
        elif rule == 'load_balance':
            # 후보 seed 확인: 아직 안 끝난 job 중에서 MLTU 기준으로 가까운 순 Top-K개 확인 (K = 10 정도) [cite: 74, 75]
            K = min(10, len(self.job_queue)) 
            
            nearest_jobs = sorted(self.job_queue, key=lambda j: np.linalg.norm(np.array(ctu.pos) - np.array(j['pos'])))
            candidate_seeds = nearest_jobs[:K]
            
            best_batch = []
            min_eta = float('inf') 
            
            # Seed 마다 nearest로 batch 만들기 [cite: 76]
            for seed in candidate_seeds:
                temp_queue = sorted(self.job_queue, key=lambda j: np.linalg.norm(np.array(seed['pos']) - np.array(j['pos'])))
                
                # 최대 7개 box 로드 [cite: 46]
                temp_batch = temp_queue[:TOTE_CAPACITY]
                
                # Batch 별 예상종료시간(ETA, Estimated Time of Arrival) 계산 (batch 내부 이동 시간) [cite: 77]
                eta = 0
                current_pos = ctu.pos 
                
                for job in temp_batch:
                    dx = abs(job['pos'][0] - current_pos[0])
                    dy = abs(job['pos'][1] - current_pos[1])
                    dz = abs(job['pos'][2] - current_pos[2])
                    
                    # 로봇 이동 시간 산출 (XY 평면 속도 및 리프팅 속도 고려) [cite: 47]
                    time_to_job = max((dx + dy) / CTU_SPEED_XY, dz / CTU_SPEED_Z)
                    eta += time_to_job
                    current_pos = job['pos'] 
                
                # ETA가 가장 작은 batch 선택하여 할당 [cite: 78]
                if eta < min_eta:
                    min_eta = eta
                    best_batch = temp_batch
            
            # 최적 배치를 찾았으므로 전체 대기열(job_queue)에서 제거 후 반환
            for job in best_batch:
                self.job_queue.remove(job)
                
            return best_batch

# ==========================================
# Slotting Rule
# 수정/검토사항: 시뮬레이션 시작 전, 물건을 창고의 어느 위치에 둘지 결정 -> 재배치 rule?
# ==========================================
def generate_fixed_inventory(rule_type, bom_df, full_plan_df):
    inventory = {} # 부품 위치 저장용 딕셔너리 생성
    all_parts = sorted(bom_df['part_code'].unique()) # 모든 부품 코드 정렬
    
    # 1. Random Slotting
    if rule_type == 'random':
        for part in all_parts:
            # 통로(X), 랙(Y), 선반(Z) 모두 random 선택
            inventory[part] = [random.choice(AISLE_X), round(random.uniform(RACK_START_Y, RACK_END_Y), 2), random.choice(SHELF_HEIGHTS)]
    
    # 2. ABC Slotting (수정/검토사항: 출고 많은 부품 입구 근처 -> 스테이션와 가까운 통로 A02/A05 입구 근처)
    elif rule_type == 'abc':
        # 부품 등장 빈도 계산
        freq = full_plan_df['부품코드'].value_counts().to_dict()
        # 빈도가 높은 순서대로 부품 리스트 정렬
        sorted_parts = sorted(all_parts, key=lambda x: freq.get(x, 0), reverse=True)

        # 부품 위치 결정
        for i, part in enumerate(sorted_parts):
            # Y를 중요도 순으로 배치
            # 순위가 높을수록(i가 작을수록) Y값이 작음(입구 쪽)
            y_pos = RACK_START_Y + (i / len(all_parts)) * (RACK_END_Y - RACK_START_Y)
            # Z좌표 index 결정
            # 순위가 높을수록 Z값이 낮음(꺼내기 쉬운 아래층)
            z_idx = min(int((i / len(all_parts)) * len(SHELF_HEIGHTS)), len(SHELF_HEIGHTS)-1)

            # 위치 저장
            inventory[part] = [random.choice(AISLE_X), round(y_pos, 2), SHELF_HEIGHTS[z_idx]]

    # 3. Characteristic Slotting
    elif rule_type == 'characteristic':
        for i, part in enumerate(all_parts):
            # 부품 리스트 순서대로 통로를 번갈아가며(Mod연산) 배치하여 혼잡도 분산
            aisle = AISLE_X[i % len(AISLE_X)]
            inventory[part] = [aisle, round(random.uniform(RACK_START_Y, RACK_END_Y), 2), random.choice(SHELF_HEIGHTS)]

    # 4. Kitting Slotting (같이 나가는 건 같이 둠)
    elif rule_type == 'kitting':
        # 주문 번호(No.)별로 묶어서 같이 주문된 부품 리스트를 추출
        order_groups = full_plan_df.groupby('No.')['부품코드'].apply(list)
        pairs = []
        for items in order_groups:
            # 2개 이상 주문된 건에 대해 모든 쌍(Pair)을 구함
            if len(items) > 1:
                pairs.extend(list(combinations(sorted(list(set(items))), 2)))
        
        # 부품쌍의 등장 빈도 계산(많이 같이 등장한 조합 찾기)
        pair_counts = Counter(pairs)
        
        # 연관성 높은 부품부터 순서대로 리스트 생성
        related_parts = []
        for (p1, p2), count in pair_counts.most_common():
            if p1 not in related_parts: related_parts.append(p1)
            if p2 not in related_parts: related_parts.append(p2)
        # 연관성 데이터가 없는 나머지 부품들도 뒤에 추가
        for p in all_parts:
            if p not in related_parts: related_parts.append(p)

        # 연관된 것끼리 같은 통로에 모으는 로직
        # 통로별 배치 수 계산
        parts_per_aisle = len(all_parts) // len(AISLE_X) + 1
        # 위치 배치
        for i, part in enumerate(related_parts):
            aisle_idx = i // parts_per_aisle
            aisle = AISLE_X[min(aisle_idx, len(AISLE_X)-1)]
            
            # 같은 통로 내에서도 인접하게 배치
            y_ratio = (i % parts_per_aisle) / parts_per_aisle
            y_pos = RACK_START_Y + y_ratio * (RACK_END_Y - RACK_START_Y)
            
            inventory[part] = [aisle, round(y_pos, 2), random.choice(SHELF_HEIGHTS)]
    
    return inventory

# ==========================================
# 하루 단위 시뮬레이션 실행
# 특정 날짜, 특정 규칙을 적용하여 하루치 작업을 시뮬레이션
# ==========================================
def run_simulation(date, inventory, dispatch_rule, date_plan):
    env = simpy.Environment() # 시뮬레이션 환경 생성
    warehouse = Warehouse(env, inventory, date_plan) # 창고 객체 생성
    
    # CTU 2대 생성 (각각 독립적인 프로세스로 동작 시작)
    ctus = [CTU(env, i, warehouse, dispatch_rule) for i in range(2)]
    
    # Generator
    def job_gen(env, wh):
        # 출고계획 데이터(plan_df)를 주문번호(No.)별로 그룹화
        for _, group in wh.plan_df.groupby('No.'):
            # 주문과 주문 사이에 2~5초 랜덤 간격 발생 (주문 유입 시뮬레이션) // 수정/검토사항: 우리 과제에서 필요한 부분인가?
            yield env.timeout(random.uniform(2, 5))
            
            # 해당 주문에 포함된 모든 부품을 큐에 추가
            for _, row in group.iterrows():
                p_id = row['부품코드']
                if p_id in wh.inventory:
                    # 큐에 작업 추가 (부품ID, 위치, 발생시간)
                    wh.job_queue.append({'item_id': p_id, 'pos': wh.inventory[p_id], 'time': env.now})

    # 종료 조건 확인용 모니터링 함수
    def monitor(env, warehouse, gen_proc, ctus):
        # 1. 모든 주문이 생성될 때까지 대기 (job_gen 프로세스기 완전히 완료될 때까지 대기)
        # 더 이상 새로운 주문이 생성되지 않을 때까지 대기하는 역할
        yield gen_proc
        
        # 2. 주문 생성이 끝났어도, 큐에 잔여 작업이 있거나 하나의 CTU라도 작업 중(busy=True)이면 반복 실행
        while warehouse.job_queue or any(c.busy for c in ctus):
            yield env.timeout(1) # 1초 단위로 시스템 상태를 계속 확인하도록 시간 진행

    # job_gen() 함수(작업 생성 프로세스)를 시뮬레이션 환경에 등록
    # 주문 발생 프로세스 실행
    gen_proc = env.process(job_gen(env, warehouse))

    # 시뮬레이션 실행 시작
    # 모니터링 프로세스가 끝날 때까지 시뮬레이션 전체 실행 (Run until done)
    # 모든 주문 생성 완료/작업 큐 완전히 비어 있음/모든 CTU가 idle 상태
    # 위 세 조건이 만족될 때까지 env.run() 계속 동작
    env.run(until=env.process(monitor(env, warehouse, gen_proc, ctus)))
    
    # 시뮬레이션 종료 후 결과(날짜, 소요시간, 총 이동거리) 반환
    # Time: 전체 작업이 완료될 때까지 소요된 총 시뮬레이션 시간, env.now는 현재 시뮬레이션 시간을 의미
    # Distance: 모든 CTU가 이동한 거리의 총합 계산, 각 CTU 객체의 total_distance 값 합산
    return {'Date': date, 'Time': round(env.now, 1), 'Distance': round(sum(c.total_distance for c in ctus), 1)}

# ==========================================
# main 함수
# ==========================================
def main():
    try:
        # outbound_plan.csv 파일을 읽어 전체 출고계획 데이터를 DataFrame으로 저장
        plan_df_full = pd.read_csv('outbound_plan.csv')
        # BOM 데이터 생성
        # 출고계획 데이터에 등장하는 모든 부품코드를 unique()로 중복 제거 후 part_code 컬럼을 가지는 DataFrame 생성
        bom_df = pd.DataFrame({'part_code': plan_df_full['부품코드'].unique()})
    except Exception as e:
        print(f"파일을 읽는 중 오류가 발생했습니다: {e}")
        return

    # 출고계획 데이터에 존재하는 날짜 목록 추출 및 정렬
    unique_dates = sorted(plan_df_full['생산일자'].unique())
    
    # 적용한 Slotting 알고리즘 목록
    slotting_rules = ['random', 'abc', 'characteristic', 'kitting'] 
    dispatch_rules = ['nearest', 'density', 'load_balance']     

    all_results = [] # 모든 시뮬레이션 결과 저장 리스트
    print(f"총 {len(unique_dates)}일치 데이터에 대해 시뮬레이션을 시작합니다.")
    
    # 모든 경우의 수(Grid Search) 실행
    # 1. Slotting rule 변경
    for s_rule in slotting_rules:
        # 수정/검토사항: 전날 부품 위치 어떻게?
        # 각 Slotting 규칙별로 인벤토리 위치를 한 번 생성
        # 생성된 위치는 모든 날짜 시뮬레이션에 공통 사용
        fixed_inventory = generate_fixed_inventory(s_rule, bom_df, plan_df_full)
        
        # 2. Dispatching rule 변경
        for d_rule in dispatch_rules:
            # 3. 날짜별 시뮬레이션 실행
            for date in unique_dates:
                # 해당 날짜 데이터만 필터링
                date_plan = plan_df_full[plan_df_full['생산일자'] == date].copy()
                
                # 시뮬레이션 1회 실행
                res = run_simulation(date, fixed_inventory, d_rule, date_plan)
                
                # 결과 딕셔너리에 사용된 Slotting + Dispatching Rule 정보 추가
                res.update({'Slotting': s_rule, 'Dispatching': d_rule})
                # 최종 결과 리스트에 저장
                all_results.append(res)

    # 결과 분석 및 출력
    df_res = pd.DataFrame(all_results) # 모든 결과를 DataFrame 형태로 변환
    
    # 피벗 테이블 생성: 행 = Rule 조합, 열 = 날짜, 값 = 소요시간
    pivoted_time = df_res.pivot_table(index=['Slotting', 'Dispatching'], columns='Date', values='Time')
    
    # Rule 조합의 날짜별 총 소요시간 합계 계산
    pivoted_time['Total'] = pivoted_time.sum(axis=1)
    
    # 시간이 적게 걸린 순서로 정렬
    pivoted_time = pivoted_time.sort_values(by='Total')

    print("\n" + "="*95)
    print(" [최종 시뮬레이션 결과] 알고리즘 조합별 일자별 런타임 요약")
    print("-" * 95)
    print(pivoted_time.to_string()) # 전체 결과 출력
    print("="*95)
    
    # 최적 알고리즘 조합 추출 및 출력
    best = pivoted_time.index[0]
    print(f"최적 알고리즘 조합: Slotting[{best[0]}] + Dispatching[{best[1]}]")

# 현재 파일이 직접 실행될 때만 main() 함수 호출
if __name__ == "__main__":
    main()

In [ ]:
import simpy
import pandas as pd         
import numpy as np          
import random               
from itertools import combinations  # 주문 내에서 함께 출고되는 부품쌍 생성 (Kitting Slotting에 사용)
from collections import Counter     # 함께 출고되는 부품쌍의 출현 빈도 계산 (어떤 부품을 서로 가깝게 배치할지 판단하기 위함)
import matplotlib.pyplot as plt     # [수정] 시각화 및 스타일링을 위한 라이브러리

# --- 시스템 환경 설정 (상수 정의) ---
# 창고의 물리적 레이아웃과 물류 로봇 성능 정의
RACK_START_Y, RACK_END_Y = 4.0, 15.1       # 랙이 설치된 Y축의 시작과 끝 좌표 범위 (4.0m ~ 15.1m 사이에만 적재 가능)
AISLE_X = [1.2, 3.6, 6.0, 8.4, 10.8, 13.2] # 창고 통로(Aisle)가 위치한 X축 좌표 리스트 (수정 혹은 검토 필요: "부품은 이 통로 중 하나에 배치됨")
STATIONS = {'S01': [3.5, 0.5, 0], 'S02': [11.0, 0.5, 0]} # 작업 스테이션 위치 (x, y, z), CTU가 작업 후 반드시 돌아오는 목적지)
CTU_SPEED_XY = 1.8  # 로봇의 XY 이동 속도 (m/s)
CTU_SPEED_Z = 0.5   # 로봇의 리프트(Z) 상하 이동 속도 (m/s)
TOTE_CAPACITY = 7   # CTU가 한 번에 운반할 수 있는 최대 토트 박스 수 (수정 혹은 검토 필요: "토트 박스 용량" -> 7층 CTU가 운반할 수 있는 최대 토트 박스 수)
SHELF_HEIGHTS = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0] # 선반의 층별 높이(Z축) 리스트 (부품의 Z좌표로 사용)

# ==========================================
# CTU class 물류 로봇
# ==========================================
class CTU: 
    # __init__ 부분은 기존 코드와 동일합니다.
    def __init__(self, env, id, warehouse, dispatch_rule):
        self.env = env                  
        self.id = id                    
        self.warehouse = warehouse      
        self.dispatch_rule = dispatch_rule 
        self.pos = STATIONS['S01'] if id == 0 else STATIONS['S02'] 

        self.total_distance = 0         
        self.completed_jobs = 0         
        self.busy = False               
        self.process = env.process(self.run())

    # [수정] 목표 위치까지 이동시간 계산 함수 (맨해튼 거리 반영)
    def calculate_travel_time(self, target_pos):
        x1, y1, z1 = self.pos
        x2, y2, z2 = target_pos
        
        if x1 == x2:
            xy_dist = abs(y1 - y2)
        else:
            xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y)
            
        dz = abs(z1 - z2)
        
        # 이동 시간은 XY주행과 리프팅(Z) 중 더 오래 걸리는 시간 적용
        travel_time = max(xy_dist / CTU_SPEED_XY, dz / CTU_SPEED_Z)
        
        # 총 물리적 이동 거리 (XY 평면 주행 거리 + Z축 리프트 거리)
        travel_distance = xy_dist + dz
        
        return travel_time, travel_distance

    def run(self): 
        while True:
            if self.warehouse.job_queue:
                self.busy = True 
                batch = self.warehouse.apply_dispatching(self, self.dispatch_rule)
                
                if batch:
                    # 3. 경로 최적화 (Greedy TSP) - 맨해튼 거리 함수 적용
                    batch.sort(key=lambda j: get_manhattan_distance(self.pos, j['pos']))
                    
                    # 4. 각 상품 위치로 이동하며 피킹 수행
                    for job in batch:
                        t, d = self.calculate_travel_time(job['pos'])
                        yield self.env.timeout(t)
                        self.pos = job['pos']     
                        self.total_distance += d  
                    
                    # 5. [수정] 작업 종료 위치에 따른 스테이션 고정 할당 
                    last_job_x = self.pos[0]
                    # X좌표가 6.0 이하(A01, A02, A03)면 S01, 아니면(A04, A05, A06) S02로 복귀 
                    target_station = STATIONS['S01'] if last_job_x <= 6.0 else STATIONS['S02']
                    
                    t, d = self.calculate_travel_time(target_station)
                    yield self.env.timeout(t) 
                    self.pos = target_station           
                    self.total_distance += d  
                
                self.busy = False 
            else:
                yield self.env.timeout(1)

    # 이동 시간 및 거리 계산
    # 목표 위치까지 이동시간 계산 함수
    def calculate_travel_time(self, target_pos):
        # 현재 위치(self.pos)와 목표 위치(target_pos) 사이의 좌표 차이 -> 거리 절댓값 계산
        dx = abs(target_pos[0] - self.pos[0]) # X축 이동거리
        dy = abs(target_pos[1] - self.pos[1]) # Y축 이동거리
        dz = abs(target_pos[2] - self.pos[2]) # Z축 이동거리
        
        # 이동 시간 계산 로직:
        # 로봇은 XY 평면 이동과 Z축(리프트) 상승/하강을 '동시에' 할 수 있다고 가정
        # XY 이동 시간과 Z 이동 시간 중 더 오래 걸리는 시간이 최종 소요 시간(max)이 됩니다.
        travel_time = max((dx + dy) / CTU_SPEED_XY, dz / CTU_SPEED_Z)
        
        # [수정] 총 이동 거리: 유클리드 거리 사용
        travel_distance = np.linalg.norm([dx, dy, dz])
        
        # 이동시간 + 총 이동거리 반환
        return travel_time, travel_distance


    # CTU의 실제 작업 수행 프로세스
    def run(self): 
        while True: # 시뮬레이션이 끝날 때까지 계속 반복
            # 1. 창고에 처리해야 할 일(job_queue)이 있는지 확인
            # 처리할 작업이 존재하면 '작업중' 상태 변경
            if self.warehouse.job_queue:
                self.busy = True # 작업 시작 상태로 변경
                
                # 2. 작업 할당 규칙에 따라 warehouse에 작업 묶음(batch)을 요청
                # 어떤 작업을 가져올지는 dispatch_rule에 따라 창고가 결정
                batch = self.warehouse.apply_dispatching(self, self.dispatch_rule)
                
                if batch:
                    # 3. 경로 최적화 (Greedy TSP)
                    # 할당받은 작업(batch)들 중, 현재 위치 기준 가까운 순서대로 방문 순서를 정렬
                    # np.linalg.norm: 벡터 간 유클리드 거리를 계산하는 함수
                    batch.sort(key=lambda j: np.linalg.norm(np.array(self.pos) - np.array(j['pos'])))
                    
                    # 4. 각 상품 위치로 이동하며 피킹 수행
                    for job in batch:
                        # 목표 위치까지 걸리는 시간(t)과 거리(d) 계산
                        t, d = self.calculate_travel_time(job['pos'])
                        
                        # "yield env.timeout(t)"
                        # 로봇이 이동하는 시간(t초) 동안 프로세스를 멈추고 대기
                        # 실제 이동시간만큼 시뮬레이션 시간 흐름 발생
                        yield self.env.timeout(t)
                        
                        self.pos = job['pos']     # 로봇 위치를 목표 지점으로 업데이트
                        self.total_distance += d  # 이동 거리 누적
                    
                    # 5. 모든 상품 피킹 완료 후, 가까운 스테이션으로 복귀
                    # d1: 현재 위치에서 S01까지의 거리
                    d1 = np.linalg.norm(np.array(self.pos[:2]) - np.array(STATIONS['S01'][:2]))
                    # d2: 현재 위치에서 S02까지의 거리
                    d2 = np.linalg.norm(np.array(self.pos[:2]) - np.array(STATIONS['S02'][:2]))
                    
                    # 더 가까운 스테이션 선택
                    near = STATIONS['S01'] if d1 < d2 else STATIONS['S02']
                    
                    # 선택된 스테이션까지 이동시간 및 거리 계산
                    t, d = self.calculate_travel_time(near)
                    yield self.env.timeout(t) # 이동 시간만큼 시뮬레이션 시간 흐름 
                    self.pos = near           # 위치 업데이트
                    self.total_distance += d  # 이동거리 누적
                
                self.busy = False # 작업 완료 후 상태를 대기중(idle)으로 변경
            
            else:
                # 6. 할 일이 없으면 1초 동안 대기했다가 다시 큐를 확인 (Busy Waiting 방지)
                yield self.env.timeout(1)


# ==========================================
# 창고 시스템 클래스 정의
# 재고 위치 및 작업 큐 관리 역할
# ==========================================
class Warehouse:
    def __init__(self, env, inventory, plan_df):
        self.env = env             # 시뮬레이션 환경 저장
        self.inventory = inventory # {부품코드: [x, y, z]} 형태의 딕셔너리 (부품별 저장 위치 정보, Slotting 결과)
        self.plan_df = plan_df     # 해당 날짜에 처리해야 할 출고 계획 데이터프레임
        self.job_queue = []        # 실시간으로 들어오는 주문들이 쌓이는 대기열, 아직 처리되지 않은 Picking 작업 큐

# dispatching rule에 따라 CTU에게 할당할 작업 배치 생성 
    def apply_dispatching(self, ctu, rule):
        # 대기 중인 작업이 없으면 빈 리스트 반환
        if not self.job_queue: return []

        # ==========================================
        # 1. Nearest-seeded batch [cite: 64]
        # ==========================================
        if rule == 'nearest':
            # 현재 위치에서 가장 물리적 거리가 가까운 작업 순서로 정렬 (Nearest seed job 하나 선택)
            self.job_queue.sort(key=lambda j: np.linalg.norm(np.array(ctu.pos) - np.array(j['pos'])))
            
            # 첫 번째 작업(Nearest seed)을 기준으로 삼음
            batch = [self.job_queue.pop(0)] 
            
            if self.job_queue:
                # Seed 주변에서 경로 상 가까운 job을 추가하기 위해 다시 정렬 
                self.job_queue.sort(key=lambda j: np.linalg.norm(np.array(batch[0]['pos']) - np.array(j['pos'])))
                
                # 한 번에 최대 7개 box 로드 가능
                extra_count = min(len(self.job_queue), TOTE_CAPACITY - 1)
                
                # Q개가 될 때까지 추가하여 할당 및 큐에서 제거 
                batch.extend(self.job_queue[:extra_count])
                del self.job_queue[:extra_count]
                
            return batch

        # ==========================================
        # 2. Density-seeded batch 
        # ==========================================
        elif rule == 'density':
            # 후보 seed 확인: 아직 안 끝난 job 중에서 MLTU 기준으로 가까운 순 Top-K개 확인 (K = 10 정도) 
            K = min(10, len(self.job_queue)) 
            candidate_seeds = sorted(self.job_queue, key=lambda j: np.linalg.norm(np.array(ctu.pos) - np.array(j['pos'])))[:K]
            
            best_seed = None
            max_density = -1
            r_y = 2.65 # 1개 베이 길이 (m) 적용 (업라이트 및 박스 여유 공간 50mm 포함된 추정 길이) 
            
            # Seed 마다 밀집도 계산 [cite: 70]
            for seed in candidate_seeds:
                density = 0
                for job in self.job_queue:
                    # 1. 동일 복도(Aisle)에 있는지 확인하여 랙 관통 탐색을 철저히 방지 (폭 1200mm 제약 고려) 
                    if job['pos'][0] == seed['pos'][0]:
                        # 2. 같은 복도라면, Seed 주변 반경 r안에 대기 job이 몇 개 있는지 세서 밀집도 정량화 
                        dy = abs(job['pos'][1] - seed['pos'][1])
                        if dy <= r_y:
                            density += 1
                
                # 가장 밀집도가 큰 seed 갱신 
                if density > max_density:
                    max_density = density
                    best_seed = seed
                    
            # 가장 밀집도가 큰 seed를 선택하여 nearest로 batch 만들어 할당 
            # 선택된 best_seed를 기준으로 전체 대기열을 가까운 순으로 정렬
            self.job_queue.sort(key=lambda j: np.linalg.norm(np.array(best_seed['pos']) - np.array(j['pos'])))
            
            # CTU가 한 번에 로드 가능한 최대 7개 할당 
            batch_size = min(len(self.job_queue), TOTE_CAPACITY)
            best_batch = self.job_queue[:batch_size]
            
            # 할당된 작업들은 큐에서 일괄 제거
            del self.job_queue[:batch_size]
            
            return best_batch

        # ==========================================
        # 3. Load-balance batch 
        # ==========================================
        elif rule == 'load_balance':
            # 후보 seed 확인: 아직 안 끝난 job 중에서 MLTU 기준으로 가까운 순 Top-K개 확인 (K = 10 정도) 
            K = min(10, len(self.job_queue)) 
            
            nearest_jobs = sorted(self.job_queue, key=lambda j: np.linalg.norm(np.array(ctu.pos) - np.array(j['pos'])))
            candidate_seeds = nearest_jobs[:K]
            
            best_batch = []
            min_eta = float('inf') 
            
            # Seed 마다 nearest로 batch 만들기
            for seed in candidate_seeds:
                temp_queue = sorted(self.job_queue, key=lambda j: np.linalg.norm(np.array(seed['pos']) - np.array(j['pos'])))
                
                # 최대 7개 box 로드 
                temp_batch = temp_queue[:TOTE_CAPACITY]
                
                # Batch 별 예상종료시간(ETA, Estimated Time of Arrival) 계산 (batch 내부 이동 시간)
                eta = 0
                current_pos = ctu.pos 
                
                for job in temp_batch:
                    dx = abs(job['pos'][0] - current_pos[0])
                    dy = abs(job['pos'][1] - current_pos[1])
                    dz = abs(job['pos'][2] - current_pos[2])
                    
                    # 로봇 이동 시간 산출 (XY 평면 속도 및 리프팅 속도 고려) 
                    time_to_job = max((dx + dy) / CTU_SPEED_XY, dz / CTU_SPEED_Z)
                    eta += time_to_job
                    current_pos = job['pos'] 
                
                # ETA가 가장 작은 batch 선택하여 할당 
                if eta < min_eta:
                    min_eta = eta
                    best_batch = temp_batch
            
            # 최적 배치를 찾았으므로 전체 대기열(job_queue)에서 제거 후 반환
            for job in best_batch:
                self.job_queue.remove(job)
                
            return best_batch

# ==========================================
# Slotting Rule
# 수정/검토사항: 시뮬레이션 시작 전, 물건을 창고의 어느 위치에 둘지 결정 -> 재배치 rule?
# ==========================================
def generate_fixed_inventory(rule_type, bom_df, full_plan_df):
    inventory = {} # 부품 위치 저장용 딕셔너리 생성
    all_parts = sorted(bom_df['part_code'].unique()) # 모든 부품 코드 정렬
    
    # [수정] 랜덤 고정 시드 적용 -> 결과 재현
    random.seed(1234)
    np.random.seed(1234)
    
    # 1. Random Slotting
    if rule_type == 'random':
        for part in all_parts:
            # 통로(X), 랙(Y), 선반(Z) 모두 random 선택
            inventory[part] = [random.choice(AISLE_X), round(random.uniform(RACK_START_Y, RACK_END_Y), 2), random.choice(SHELF_HEIGHTS)]
    
    # 2. ABC Slotting (수정/검토사항: 출고 많은 부품 입구 근처 -> 스테이션와 가까운 통로 A02/A05 입구 근처)
    elif rule_type == 'abc':
        # 부품 등장 빈도 계산
        freq = full_plan_df['부품코드'].value_counts().to_dict()
        # 빈도가 높은 순서대로 부품 리스트 정렬
        sorted_parts = sorted(all_parts, key=lambda x: freq.get(x, 0), reverse=True)

        # 부품 위치 결정
        for i, part in enumerate(sorted_parts):
            # Y를 중요도 순으로 배치
            # 순위가 높을수록(i가 작을수록) Y값이 작음(입구 쪽)
            y_pos = RACK_START_Y + (i / len(all_parts)) * (RACK_END_Y - RACK_START_Y)
            # Z좌표 index 결정
            # 순위가 높을수록 Z값이 낮음(꺼내기 쉬운 아래층)
            z_idx = min(int((i / len(all_parts)) * len(SHELF_HEIGHTS)), len(SHELF_HEIGHTS)-1)

            # 위치 저장
            inventory[part] = [random.choice(AISLE_X), round(y_pos, 2), SHELF_HEIGHTS[z_idx]]

    # 3. Characteristic Slotting
    elif rule_type == 'characteristic':
        for i, part in enumerate(all_parts):
            # 부품 리스트 순서대로 통로를 번갈아가며(Mod연산) 배치하여 혼잡도 분산
            aisle = AISLE_X[i % len(AISLE_X)]
            inventory[part] = [aisle, round(random.uniform(RACK_START_Y, RACK_END_Y), 2), random.choice(SHELF_HEIGHTS)]

    # 4. Kitting Slotting (같이 나가는 건 같이 둠)
    elif rule_type == 'kitting':
        # 주문 번호(No.)별로 묶어서 같이 주문된 부품 리스트를 추출
        order_groups = full_plan_df.groupby('No.')['부품코드'].apply(list)
        pairs = []
        for items in order_groups:
            # 2개 이상 주문된 건에 대해 모든 쌍(Pair)을 구함
            if len(items) > 1:
                pairs.extend(list(combinations(sorted(list(set(items))), 2)))
        
        # 부품쌍의 등장 빈도 계산(많이 같이 등장한 조합 찾기)
        pair_counts = Counter(pairs)
        
        # 연관성 높은 부품부터 순서대로 리스트 생성
        related_parts = []
        for (p1, p2), count in pair_counts.most_common():
            if p1 not in related_parts: related_parts.append(p1)
            if p2 not in related_parts: related_parts.append(p2)
        # 연관성 데이터가 없는 나머지 부품들도 뒤에 추가
        for p in all_parts:
            if p not in related_parts: related_parts.append(p)

        # 연관된 것끼리 같은 통로에 모으는 로직
        # 통로별 배치 수 계산
        parts_per_aisle = len(all_parts) // len(AISLE_X) + 1
        # 위치 배치
        for i, part in enumerate(related_parts):
            aisle_idx = i // parts_per_aisle
            aisle = AISLE_X[min(aisle_idx, len(AISLE_X)-1)]
            
            # 같은 통로 내에서도 인접하게 배치
            y_ratio = (i % parts_per_aisle) / parts_per_aisle
            y_pos = RACK_START_Y + y_ratio * (RACK_END_Y - RACK_START_Y)
            
            inventory[part] = [aisle, round(y_pos, 2), random.choice(SHELF_HEIGHTS)]
    
    return inventory

# ==========================================
# 하루 단위 시뮬레이션 실행
# 특정 날짜, 특정 규칙을 적용하여 하루치 작업을 시뮬레이션
# ==========================================
def run_simulation(date, inventory, dispatch_rule, date_plan):
    env = simpy.Environment() # 시뮬레이션 환경 생성
    warehouse = Warehouse(env, inventory, date_plan) # 창고 객체 생성
    
    # CTU 2대 생성 (각각 독립적인 프로세스로 동작 시작)
    ctus = [CTU(env, i, warehouse, dispatch_rule) for i in range(2)]
    
    # Generator
    def job_gen(env, wh):
        # 출고계획 데이터(plan_df)를 주문번호(No.)별로 그룹화
        for _, group in wh.plan_df.groupby('No.'):
            # 주문과 주문 사이에 2~5초 랜덤 간격 발생 (주문 유입 시뮬레이션) // 수정/검토사항: 우리 과제에서 필요한 부분인가?
            yield env.timeout(random.uniform(2, 5))
            
            # 해당 주문에 포함된 모든 부품을 큐에 추가
            for _, row in group.iterrows():
                p_id = row['부품코드']
                if p_id in wh.inventory:
                    # 큐에 작업 추가 (부품ID, 위치, 발생시간)
                    wh.job_queue.append({'item_id': p_id, 'pos': wh.inventory[p_id], 'time': env.now})

    # 종료 조건 확인용 모니터링 함수
    def monitor(env, warehouse, gen_proc, ctus):
        # 1. 모든 주문이 생성될 때까지 대기 (job_gen 프로세스기 완전히 완료될 때까지 대기)
        # 더 이상 새로운 주문이 생성되지 않을 때까지 대기하는 역할
        yield gen_proc
        
        # 2. 주문 생성이 끝났어도, 큐에 잔여 작업이 있거나 하나의 CTU라도 작업 중(busy=True)이면 반복 실행
        while warehouse.job_queue or any(c.busy for c in ctus):
            yield env.timeout(1) # 1초 단위로 시스템 상태를 계속 확인하도록 시간 진행

    # job_gen() 함수(작업 생성 프로세스)를 시뮬레이션 환경에 등록
    # 주문 발생 프로세스 실행
    gen_proc = env.process(job_gen(env, warehouse))

    # 시뮬레이션 실행 시작
    # 모니터링 프로세스가 끝날 때까지 시뮬레이션 전체 실행 (Run until done)
    # 모든 주문 생성 완료/작업 큐 완전히 비어 있음/모든 CTU가 idle 상태
    # 위 세 조건이 만족될 때까지 env.run() 계속 동작
    env.run(until=env.process(monitor(env, warehouse, gen_proc, ctus)))
    
    # 시뮬레이션 종료 후 결과(날짜, 소요시간, 총 이동거리) 반환
    # Time: 전체 작업이 완료될 때까지 소요된 총 시뮬레이션 시간, env.now는 현재 시뮬레이션 시간을 의미
    # Distance: 모든 CTU가 이동한 거리의 총합 계산, 각 CTU 객체의 total_distance 값 합산
    return {'Date': date, 'Time': round(env.now, 1), 'Distance': round(sum(c.total_distance for c in ctus), 1)}

# ==========================================
# main 함수
# ==========================================
def main():
    try:
        # outbound_plan.csv 파일을 읽어 전체 출고계획 데이터를 DataFrame으로 저장
        plan_df_full = pd.read_csv('outbound_plan.csv')
        # BOM 데이터 생성
        # 출고계획 데이터에 등장하는 모든 부품코드를 unique()로 중복 제거 후 part_code 컬럼을 가지는 DataFrame 생성
        bom_df = pd.DataFrame({'part_code': plan_df_full['부품코드'].unique()})
    except Exception as e:
        print(f"파일을 읽는 중 오류가 발생했습니다: {e}")
        return

    # 출고계획 데이터에 존재하는 날짜 목록 추출 및 정렬
    unique_dates = sorted(plan_df_full['생산일자'].unique())
    
    # 적용한 Slotting 알고리즘 목록
    slotting_rules = ['random', 'abc', 'characteristic', 'kitting'] 
    dispatch_rules = ['nearest', 'density', 'load_balance']     

    all_results = [] # 모든 시뮬레이션 결과 저장 리스트
    print(f"총 {len(unique_dates)}일치 데이터에 대해 시뮬레이션을 시작합니다.")
    
    # 모든 경우의 수(Grid Search) 실행
    # 1. Slotting rule 변경
    for s_rule in slotting_rules:
        # 수정/검토사항: 전날 부품 위치 어떻게?
        # 각 Slotting 규칙별로 인벤토리 위치를 한 번 생성
        # 생성된 위치는 모든 날짜 시뮬레이션에 공통 사용
        fixed_inventory = generate_fixed_inventory(s_rule, bom_df, plan_df_full)
        
        # 2. Dispatching rule 변경
        for d_rule in dispatch_rules:
            # 3. 날짜별 시뮬레이션 실행
            for date in unique_dates:
                # 해당 날짜 데이터만 필터링
                date_plan = plan_df_full[plan_df_full['생산일자'] == date].copy()
                
                # 시뮬레이션 1회 실행
                res = run_simulation(date, fixed_inventory, d_rule, date_plan)
                
                # 결과 딕셔너리에 사용된 Slotting + Dispatching Rule 정보 추가
                res.update({'Slotting': s_rule, 'Dispatching': d_rule})
                # 최종 결과 리스트에 저장
                all_results.append(res)

    # 결과 분석 및 출력
    df_res = pd.DataFrame(all_results) # 모든 결과를 DataFrame 형태로 변환
    
    # 피벗 테이블 생성: 행 = Rule 조합, 열 = 날짜, 값 = 소요시간
    pivoted_time = df_res.pivot_table(index=['Slotting', 'Dispatching'], columns='Date', values='Time')
    
    # Rule 조합의 날짜별 총 소요시간 합계 계산
    pivoted_time['Total'] = pivoted_time.sum(axis=1)
    
    # 시간이 적게 걸린 순서로 정렬
    pivoted_time = pivoted_time.sort_values(by='Total')

    # =========================
    # --- 결과 표 & 그래프 ---
    # =========================
    print("\n" + "="*95)
    print(" [최종 시뮬레이션 결과] 알고리즘 조합별 일자별 런타임 요약")
    print("-" * 95)
    
    # [수정] pandas 스타일링 표 
    try:
        from IPython.display import display
        styled = pivoted_time.style.background_gradient(cmap='YlGnBu', subset=pivoted_time.columns[:-1]) \
            .highlight_min(subset=['Total'], color='lightgreen').format("{:.1f}")
        display(styled)
    except:
        print(pivoted_time.to_string())  # 텍스트 출력 fallback
        
    print("="*95)
    
    # 최적 알고리즘 조합 추출 및 출력
    best = pivoted_time.index[0]
    print(f"최적 알고리즘 조합: Slotting[{best[0]}] + Dispatching[{best[1]}]")

# 현재 파일이 직접 실행될 때만 main() 함수 호출
if __name__ == "__main__":
    main()

In [ ]:
import simpy
import pandas as pd         
import numpy as np          
import random               
from itertools import combinations  # 주문 내에서 함께 출고되는 부품쌍 생성 (Kitting Slotting에 사용)
from collections import Counter     # 함께 출고되는 부품쌍의 출현 빈도 계산 (어떤 부품을 서로 가깝게 배치할지 판단하기 위함)
import matplotlib.pyplot as plt     # [수정] 시각화 및 스타일링을 위한 라이브러리

# --- 시스템 환경 설정 (상수 정의) ---
# 창고의 물리적 레이아웃과 물류 로봇 성능 정의
RACK_START_Y, RACK_END_Y = 4.0, 15.1       # 랙이 설치된 Y축의 시작과 끝 좌표 범위 (4.0m ~ 15.1m 사이에만 적재 가능)
AISLE_X = [1.2, 3.6, 6.0, 8.4, 10.8, 13.2] # 창고 통로(Aisle)가 위치한 X축 좌표 리스트 (수정 혹은 검토 필요: "부품은 이 통로 중 하나에 배치됨")
STATIONS = {'S01': [3.5, 0.5, 0], 'S02': [11.0, 0.5, 0]} # 작업 스테이션 위치 (x, y, z), CTU가 작업 후 반드시 돌아오는 목적지)
CTU_SPEED_XY = 1.8  # 로봇의 XY 이동 속도 (m/s)
CTU_SPEED_Z = 0.5   # 로봇의 리프트(Z) 상하 이동 속도 (m/s)
TOTE_CAPACITY = 7   # CTU가 한 번에 운반할 수 있는 최대 토트 박스 수 (수정 혹은 검토 필요: "토트 박스 용량" -> 7층 CTU가 운반할 수 있는 최대 토트 박스 수)
SHELF_HEIGHTS = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0] # 선반의 층별 높이(Z축) 리스트 (부품의 Z좌표로 사용)

# [수정] 교차 통로(Cross Aisle) 설정
# 로봇이 다른 통로로 넘어갈 때 랙을 관통하지 않고 앞쪽으로 빠져나와서 이동하는 기준 Y 좌표
CROSS_AISLE_Y = 2.0 

# ==========================================
# [신규] 창고 레이아웃 기반 맨해튼 직교 거리 계산 함수
# ==========================================
def get_manhattan_distance(pos1, pos2):
    x1, y1, z1 = pos1
    x2, y2, z2 = pos2
    
    # 1. 같은 통로(X좌표 동일)일 경우: 단순히 Y축 깊이 차이만 계산
    if x1 == x2:
        xy_dist = abs(y1 - y2)
    # 2. 다른 통로일 경우: 현재 통로에서 교차 통로(Y=2.0)로 직진하여 빠져나온 뒤, 가로(X)로 이동 후 새 통로로 진입
    else:
        xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y)
        
    # Z축(높이) 차이 포함하여 최종 3D 맨해튼 거리 반환 (의사결정 우선순위용)
    return xy_dist + abs(z1 - z2)

# ==========================================
# CTU class 물류 로봇
# ==========================================
class CTU: # 물류 로봇 정의
    def __init__(self, env, id, warehouse, dispatch_rule): # env: 시뮬레이션 환경, id: CTU 번호, warehouse: 작업정보 접근, dispatch_rule: 작업 할당 알고리즘)
        self.env = env                  # Simpy 시뮬레이션 환경 객체 (현재 시뮬레이션 시간을 알고 있음)
        self.id = id                    # 로봇의 고유 번호 (0, 1, 2...)
        self.warehouse = warehouse      # 로봇이 소속된 창고 객체
        self.dispatch_rule = dispatch_rule # 이 로봇에게 적용될 작업 할당 알고리즘 (nearest, density, load_balance)
        
        # 초기 로봇 위치 설정: 로봇 2대가 서로 다른 스테이션에서 시작하도록 설정.
        self.pos = STATIONS['S01'] if id == 0 else STATIONS['S02'] # 0번 CTU는 S01에서 시작, 1번 CTU는 S02에서 시작

        # 총 이동거리 기록
        # 처리 작업 수 기록
        self.total_distance = 0         # 시뮬레이션 동안 로봇이 이동한 총 누적 거리
        self.completed_jobs = 0         # 처리한 총 작업 수 (현재 코드에서는 집계만 하고 사용 안 함)
        self.busy = False               # 로봇이 현재 작업 중인지 여부 (True: 작업중, False: 대기중)
        
        # 핵심: env.process(self.run())
        # CTU의 작업 수행 프로세스를 시뮬레이션에 등록
        # run() 함수가 시간 흐름에 따라 실행
        # 이 객체가 생성되는 순간, self.run() 메서드를 SimPy의 프로세스로 등록
        # 시뮬레이션이 시작(env.run)되면 로봇은 독립적으로 움직임.
        self.process = env.process(self.run())

    # 이동 시간 및 거리 계산
    # 목표 위치까지 이동시간 계산 함수
    def calculate_travel_time(self, target_pos):
        # [수정] 현재 위치(self.pos)와 목표 위치(target_pos) 사이의 거리를 맨해튼 직교 거리로 계산
        x1, y1, z1 = self.pos
        x2, y2, z2 = target_pos
        
        if x1 == x2:
            xy_dist = abs(y1 - y2)
        else:
            xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y)
            
        dz = abs(z1 - z2)
        
        # 이동 시간 계산 로직:
        # 로봇은 XY 평면 이동과 Z축(리프트) 상승/하강을 '동시에' 할 수 있다고 가정
        # XY 이동 시간과 Z 이동 시간 중 더 오래 걸리는 시간이 최종 소요 시간(max)이 됩니다.
        travel_time = max(xy_dist / CTU_SPEED_XY, dz / CTU_SPEED_Z)
        
        # [수정] 총 이동 거리: 맨해튼 궤적 기반 실제 주행 거리 산출
        travel_distance = xy_dist + dz
        
        # 이동시간 + 총 이동거리 반환
        return travel_time, travel_distance


    # CTU의 실제 작업 수행 프로세스
    def run(self): 
        while True: # 시뮬레이션이 끝날 때까지 계속 반복
            # 1. 창고에 처리해야 할 일(job_queue)이 있는지 확인
            # 처리할 작업이 존재하면 '작업중' 상태 변경
            if self.warehouse.job_queue:
                self.busy = True # 작업 시작 상태로 변경
                
                # 2. 작업 할당 규칙에 따라 warehouse에 작업 묶음(batch)을 요청
                # 어떤 작업을 가져올지는 dispatch_rule에 따라 창고가 결정
                batch = self.warehouse.apply_dispatching(self, self.dispatch_rule)
                
                if batch:
                    # 3. 경로 최적화 (Greedy TSP)
                    # 할당받은 작업(batch)들 중, 현재 위치 기준 가까운 순서대로 방문 순서를 정렬
                    # [수정] 유클리드 대신 맨해튼 거리 산출 함수 적용
                    batch.sort(key=lambda j: get_manhattan_distance(self.pos, j['pos']))
                    
                    # 4. 각 상품 위치로 이동하며 피킹 수행
                    for job in batch:
                        # 목표 위치까지 걸리는 시간(t)과 거리(d) 계산
                        t, d = self.calculate_travel_time(job['pos'])
                        
                        # "yield env.timeout(t)"
                        # 로봇이 이동하는 시간(t초) 동안 프로세스를 멈추고 대기
                        # 실제 이동시간만큼 시뮬레이션 시간 흐름 발생
                        yield self.env.timeout(t)
                        
                        self.pos = job['pos']     # 로봇 위치를 목표 지점으로 업데이트
                        self.total_distance += d  # 이동 거리 누적
                    
                    # 5. 모든 상품 피킹 완료 후, 가까운 스테이션으로 복귀
                    # [수정] 작업이 끝난 통로(X좌표) 위치에 따라 복귀할 지정 스테이션 결정
                    last_job_x = self.pos[0]
                    # X좌표가 6.0 이하(A01, A02, A03)면 S01, 아니면(A04, A05, A06) S02로 고정 복귀
                    target_station = STATIONS['S01'] if last_job_x <= 6.0 else STATIONS['S02']
                    
                    # 선택된 스테이션까지 이동시간 및 거리 계산
                    t, d = self.calculate_travel_time(target_station)
                    yield self.env.timeout(t) # 이동 시간만큼 시뮬레이션 시간 흐름 
                    self.pos = target_station           # 위치 업데이트
                    self.total_distance += d  # 이동거리 누적
                
                self.busy = False # 작업 완료 후 상태를 대기중(idle)으로 변경
            
            else:
                # 6. 할 일이 없으면 1초 동안 대기했다가 다시 큐를 확인 (Busy Waiting 방지)
                yield self.env.timeout(1)

# ==========================================
# 창고 시스템 클래스 정의
# 재고 위치 및 작업 큐 관리 역할
# ==========================================
class Warehouse:
    def __init__(self, env, inventory, plan_df):
        self.env = env             # 시뮬레이션 환경 저장
        self.inventory = inventory # {부품코드: [x, y, z]} 형태의 딕셔너리 (부품별 저장 위치 정보, Slotting 결과)
        self.plan_df = plan_df     # 해당 날짜에 처리해야 할 출고 계획 데이터프레임
        self.job_queue = []        # 실시간으로 들어오는 주문들이 쌓이는 대기열, 아직 처리되지 않은 Picking 작업 큐

    # dispatching rule에 따라 CTU에게 할당할 작업 배치 생성 
    def apply_dispatching(self, ctu, rule):
        # 대기 중인 작업이 없으면 빈 리스트 반환
        if not self.job_queue: return []

        # ==========================================
        # [cite_start]1. Nearest-seeded batch [cite: 64]
        # ==========================================
        if rule == 'nearest':
            # 현재 위치에서 가장 물리적 거리가 가까운 작업 순서로 정렬 (Nearest seed job 하나 선택)
            # [수정] 맨해튼 직교 거리 기준 적용
            self.job_queue.sort(key=lambda j: get_manhattan_distance(ctu.pos, j['pos']))
            
            # 첫 번째 작업(Nearest seed)을 기준으로 삼음
            batch = [self.job_queue.pop(0)] 
            
            if self.job_queue:
                # Seed 주변에서 경로 상 가까운 job을 추가하기 위해 다시 정렬 
                self.job_queue.sort(key=lambda j: get_manhattan_distance(batch[0]['pos'], j['pos']))
                
                # 한 번에 최대 7개 box 로드 가능
                extra_count = min(len(self.job_queue), TOTE_CAPACITY - 1)
                
                # Q개가 될 때까지 추가하여 할당 및 큐에서 제거 
                batch.extend(self.job_queue[:extra_count])
                del self.job_queue[:extra_count]
                
            return batch

        # ==========================================
        # 2. Density-seeded batch 
        # ==========================================
        elif rule == 'density':
            # [수정] 후보 seed 확인: 아직 안 끝난 job 중에서 MLTU 기준으로 가까운 순 Top-K개 확인 (K = 10 정도) 
            K = min(10, len(self.job_queue)) 
            candidate_seeds = sorted(self.job_queue, key=lambda j: get_manhattan_distance(ctu.pos, j['pos']))[:K]
            
            best_seed = None
            max_density = -1
            r_y = 2.65 # [수정] 1개 베이 길이 (m) 적용 (업라이트 및 박스 여유 공간 50mm 포함된 추정 길이) 
            
            # [cite_start]Seed 마다 밀집도 계산 [cite: 70]
            for seed in candidate_seeds:
                density = 0
                for job in self.job_queue:
                    # 1. 동일 복도(Aisle)에 있는지 확인하여 랙 관통 탐색을 철저히 방지 (폭 1200mm 제약 고려) 
                    if job['pos'][0] == seed['pos'][0]:
                        # 2. 같은 복도라면, Seed 주변 반경 r안에 대기 job이 몇 개 있는지 세서 밀집도 정량화 
                        dy = abs(job['pos'][1] - seed['pos'][1])
                        if dy <= r_y:
                            density += 1
                
                # 가장 밀집도가 큰 seed 갱신 
                if density > max_density:
                    max_density = density
                    best_seed = seed
                    
            # 가장 밀집도가 큰 seed를 선택하여 nearest로 batch 만들어 할당 
            # 선택된 best_seed를 기준으로 전체 대기열을 가까운 순으로 정렬
            self.job_queue.sort(key=lambda j: get_manhattan_distance(best_seed['pos'], j['pos']))
            
            # CTU가 한 번에 로드 가능한 최대 7개 할당 
            batch_size = min(len(self.job_queue), TOTE_CAPACITY)
            best_batch = self.job_queue[:batch_size]
            
            # 할당된 작업들은 큐에서 일괄 제거
            del self.job_queue[:batch_size]
            
            return best_batch

        # ==========================================
        # 3. Load-balance batch 
        # ==========================================
        elif rule == 'load_balance':
            # 후보 seed 확인: 아직 안 끝난 job 중에서 MLTU 기준으로 가까운 순 Top-K개 확인 (K = 10 정도) 
            K = min(10, len(self.job_queue)) 
            
            # [수정] 맨해튼 거리 기준 상위 10개 추출
            nearest_jobs = sorted(self.job_queue, key=lambda j: get_manhattan_distance(ctu.pos, j['pos']))
            candidate_seeds = nearest_jobs[:K]
            
            best_batch = []
            min_eta = float('inf') 
            
            # Seed 마다 nearest로 batch 만들기
            for seed in candidate_seeds:
                temp_queue = sorted(self.job_queue, key=lambda j: get_manhattan_distance(seed['pos'], j['pos']))
                
                # 최대 7개 box 로드 
                temp_batch = temp_queue[:TOTE_CAPACITY]
                
                # Batch 별 예상종료시간(ETA, Estimated Time of Arrival) 계산 (batch 내부 이동 시간)
                eta = 0
                current_pos = ctu.pos 
                
                for job in temp_batch:
                    # [수정] ETA 계산 시에도 로봇의 교차 통로 이동 논리 적용
                    x1, y1, z1 = current_pos
                    x2, y2, z2 = job['pos']
                    
                    if x1 == x2:
                        xy_dist = abs(y1 - y2)
                    else:
                        xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y)
                        
                    dz = abs(z1 - z2)
                    
                    # 로봇 이동 시간 산출 (XY 평면 속도 및 리프팅 속도 고려) 
                    time_to_job = max(xy_dist / CTU_SPEED_XY, dz / CTU_SPEED_Z)
                    eta += time_to_job
                    current_pos = job['pos'] 
                
                # ETA가 가장 작은 batch 선택하여 할당 
                if eta < min_eta:
                    min_eta = eta
                    best_batch = temp_batch
            
            # 최적 배치를 찾았으므로 전체 대기열(job_queue)에서 제거 후 반환
            for job in best_batch:
                self.job_queue.remove(job)
                
            return best_batch

# ==========================================
# Slotting Rule
# 수정/검토사항: 시뮬레이션 시작 전, 물건을 창고의 어느 위치에 둘지 결정 -> 재배치 rule?
# ==========================================
def generate_fixed_inventory(rule_type, bom_df, full_plan_df):
    inventory = {} # 부품 위치 저장용 딕셔너리 생성
    all_parts = sorted(bom_df['part_code'].unique()) # 모든 부품 코드 정렬
    
    # [수정] 랜덤 고정 시드 적용 -> 결과 재현
    random.seed(1234)
    np.random.seed(1234)
    
    # 1. Random Slotting
    if rule_type == 'random':
        for part in all_parts:
            # 통로(X), 랙(Y), 선반(Z) 모두 random 선택
            inventory[part] = [random.choice(AISLE_X), round(random.uniform(RACK_START_Y, RACK_END_Y), 2), random.choice(SHELF_HEIGHTS)]
    
    # 2. ABC Slotting (수정/검토사항: 출고 많은 부품 입구 근처 -> 스테이션와 가까운 통로 A02/A05 입구 근처)
    elif rule_type == 'abc':
        # 부품 등장 빈도 계산
        freq = full_plan_df['부품코드'].value_counts().to_dict()
        # 빈도가 높은 순서대로 부품 리스트 정렬
        sorted_parts = sorted(all_parts, key=lambda x: freq.get(x, 0), reverse=True)

        # 부품 위치 결정
        for i, part in enumerate(sorted_parts):
            # Y를 중요도 순으로 배치
            # 순위가 높을수록(i가 작을수록) Y값이 작음(입구 쪽)
            y_pos = RACK_START_Y + (i / len(all_parts)) * (RACK_END_Y - RACK_START_Y)
            # Z좌표 index 결정
            # 순위가 높을수록 Z값이 낮음(꺼내기 쉬운 아래층)
            z_idx = min(int((i / len(all_parts)) * len(SHELF_HEIGHTS)), len(SHELF_HEIGHTS)-1)

            # 위치 저장
            inventory[part] = [random.choice(AISLE_X), round(y_pos, 2), SHELF_HEIGHTS[z_idx]]

    # 3. Characteristic Slotting
    elif rule_type == 'characteristic':
        for i, part in enumerate(all_parts):
            # 부품 리스트 순서대로 통로를 번갈아가며(Mod연산) 배치하여 혼잡도 분산
            aisle = AISLE_X[i % len(AISLE_X)]
            inventory[part] = [aisle, round(random.uniform(RACK_START_Y, RACK_END_Y), 2), random.choice(SHELF_HEIGHTS)]

    # 4. Kitting Slotting (같이 나가는 건 같이 둠)
    elif rule_type == 'kitting':
        # 주문 번호(No.)별로 묶어서 같이 주문된 부품 리스트를 추출
        order_groups = full_plan_df.groupby('No.')['부품코드'].apply(list)
        pairs = []
        for items in order_groups:
            # 2개 이상 주문된 건에 대해 모든 쌍(Pair)을 구함
            if len(items) > 1:
                pairs.extend(list(combinations(sorted(list(set(items))), 2)))
        
        # 부품쌍의 등장 빈도 계산(많이 같이 등장한 조합 찾기)
        pair_counts = Counter(pairs)
        
        # 연관성 높은 부품부터 순서대로 리스트 생성
        related_parts = []
        for (p1, p2), count in pair_counts.most_common():
            if p1 not in related_parts: related_parts.append(p1)
            if p2 not in related_parts: related_parts.append(p2)
        # 연관성 데이터가 없는 나머지 부품들도 뒤에 추가
        for p in all_parts:
            if p not in related_parts: related_parts.append(p)

        # 연관된 것끼리 같은 통로에 모으는 로직
        # 통로별 배치 수 계산
        parts_per_aisle = len(all_parts) // len(AISLE_X) + 1
        # 위치 배치
        for i, part in enumerate(related_parts):
            aisle_idx = i // parts_per_aisle
            aisle = AISLE_X[min(aisle_idx, len(AISLE_X)-1)]
            
            # 같은 통로 내에서도 인접하게 배치
            y_ratio = (i % parts_per_aisle) / parts_per_aisle
            y_pos = RACK_START_Y + y_ratio * (RACK_END_Y - RACK_START_Y)
            
            inventory[part] = [aisle, round(y_pos, 2), random.choice(SHELF_HEIGHTS)]
    
    return inventory

# ==========================================
# 하루 단위 시뮬레이션 실행
# 특정 날짜, 특정 규칙을 적용하여 하루치 작업을 시뮬레이션
# ==========================================
def run_simulation(date, inventory, dispatch_rule, date_plan):
    env = simpy.Environment() # 시뮬레이션 환경 생성
    warehouse = Warehouse(env, inventory, date_plan) # 창고 객체 생성
    
    # CTU 2대 생성 (각각 독립적인 프로세스로 동작 시작)
    ctus = [CTU(env, i, warehouse, dispatch_rule) for i in range(2)]
    
    # Generator
    def job_gen(env, wh):
        # 출고계획 데이터(plan_df)를 주문번호(No.)별로 그룹화
        for _, group in wh.plan_df.groupby('No.'):
            # 주문과 주문 사이에 2~5초 랜덤 간격 발생 (주문 유입 시뮬레이션) // 수정/검토사항: 우리 과제에서 필요한 부분인가?
            yield env.timeout(random.uniform(2, 5))
            
            # 해당 주문에 포함된 모든 부품을 큐에 추가
            for _, row in group.iterrows():
                p_id = row['부품코드']
                if p_id in wh.inventory:
                    # 큐에 작업 추가 (부품ID, 위치, 발생시간)
                    wh.job_queue.append({'item_id': p_id, 'pos': wh.inventory[p_id], 'time': env.now})

    # 종료 조건 확인용 모니터링 함수
    def monitor(env, warehouse, gen_proc, ctus):
        # 1. 모든 주문이 생성될 때까지 대기 (job_gen 프로세스기 완전히 완료될 때까지 대기)
        # 더 이상 새로운 주문이 생성되지 않을 때까지 대기하는 역할
        yield gen_proc
        
        # 2. 주문 생성이 끝났어도, 큐에 잔여 작업이 있거나 하나의 CTU라도 작업 중(busy=True)이면 반복 실행
        while warehouse.job_queue or any(c.busy for c in ctus):
            yield env.timeout(1) # 1초 단위로 시스템 상태를 계속 확인하도록 시간 진행

    # job_gen() 함수(작업 생성 프로세스)를 시뮬레이션 환경에 등록
    # 주문 발생 프로세스 실행
    gen_proc = env.process(job_gen(env, warehouse))

    # 시뮬레이션 실행 시작
    # 모니터링 프로세스가 끝날 때까지 시뮬레이션 전체 실행 (Run until done)
    # 모든 주문 생성 완료/작업 큐 완전히 비어 있음/모든 CTU가 idle 상태
    # 위 세 조건이 만족될 때까지 env.run() 계속 동작
    env.run(until=env.process(monitor(env, warehouse, gen_proc, ctus)))
    
    # 시뮬레이션 종료 후 결과(날짜, 소요시간, 총 이동거리) 반환
    # Time: 전체 작업이 완료될 때까지 소요된 총 시뮬레이션 시간, env.now는 현재 시뮬레이션 시간을 의미
    # Distance: 모든 CTU가 이동한 거리의 총합 계산, 각 CTU 객체의 total_distance 값 합산
    return {'Date': date, 'Time': round(env.now, 1), 'Distance': round(sum(c.total_distance for c in ctus), 1)}

# ==========================================
# main 함수
# ==========================================
def main():
    try:
        # outbound_plan.csv 파일을 읽어 전체 출고계획 데이터를 DataFrame으로 저장
        plan_df_full = pd.read_csv('outbound_plan.csv')
        # BOM 데이터 생성
        # 출고계획 데이터에 등장하는 모든 부품코드를 unique()로 중복 제거 후 part_code 컬럼을 가지는 DataFrame 생성
        bom_df = pd.DataFrame({'part_code': plan_df_full['부품코드'].unique()})
    except Exception as e:
        print(f"파일을 읽는 중 오류가 발생했습니다: {e}")
        return

    # 출고계획 데이터에 존재하는 날짜 목록 추출 및 정렬
    unique_dates = sorted(plan_df_full['생산일자'].unique())
    
    # 적용한 Slotting 알고리즘 목록
    slotting_rules = ['random', 'abc', 'characteristic', 'kitting'] 
    dispatch_rules = ['nearest', 'density', 'load_balance']     

    all_results = [] # 모든 시뮬레이션 결과 저장 리스트
    print(f"총 {len(unique_dates)}일치 데이터에 대해 시뮬레이션을 시작합니다.")
    
    # 모든 경우의 수(Grid Search) 실행
    # 1. Slotting rule 변경
    for s_rule in slotting_rules:
        # 수정/검토사항: 전날 부품 위치 어떻게?
        # 각 Slotting 규칙별로 인벤토리 위치를 한 번 생성
        # 생성된 위치는 모든 날짜 시뮬레이션에 공통 사용
        fixed_inventory = generate_fixed_inventory(s_rule, bom_df, plan_df_full)
        
        # 2. Dispatching rule 변경
        for d_rule in dispatch_rules:
            # 3. 날짜별 시뮬레이션 실행
            for date in unique_dates:
                # 해당 날짜 데이터만 필터링
                date_plan = plan_df_full[plan_df_full['생산일자'] == date].copy()
                
                # 시뮬레이션 1회 실행
                res = run_simulation(date, fixed_inventory, d_rule, date_plan)
                
                # 결과 딕셔너리에 사용된 Slotting + Dispatching Rule 정보 추가
                res.update({'Slotting': s_rule, 'Dispatching': d_rule})
                # 최종 결과 리스트에 저장
                all_results.append(res)

    # 결과 분석 및 출력
    df_res = pd.DataFrame(all_results) # 모든 결과를 DataFrame 형태로 변환
    
    # 피벗 테이블 생성: 행 = Rule 조합, 열 = 날짜, 값 = 소요시간
    pivoted_time = df_res.pivot_table(index=['Slotting', 'Dispatching'], columns='Date', values='Time')
    
    # Rule 조합의 날짜별 총 소요시간 합계 계산
    pivoted_time['Total'] = pivoted_time.sum(axis=1)
    
    # 시간이 적게 걸린 순서로 정렬
    pivoted_time = pivoted_time.sort_values(by='Total')

    # =========================
    # --- 결과 표 & 그래프 ---
    # =========================
    print("\n" + "="*95)
    print(" [최종 시뮬레이션 결과] 알고리즘 조합별 일자별 런타임 요약")
    print("-" * 95)
    
    # [수정] pandas 스타일링 표 
    try:
        from IPython.display import display
        styled = pivoted_time.style.background_gradient(cmap='YlGnBu', subset=pivoted_time.columns[:-1]) \
            .highlight_min(subset=['Total'], color='lightgreen').format("{:.1f}")
        display(styled)
    except:
        print(pivoted_time.to_string())  # 텍스트 출력 fallback
        
    print("="*95)
    
    # 최적 알고리즘 조합 추출 및 출력
    best = pivoted_time.index[0]
    print(f"최적 알고리즘 조합: Slotting[{best[0]}] + Dispatching[{best[1]}]")

# 현재 파일이 직접 실행될 때만 main() 함수 호출
if __name__ == "__main__":
    main()